# NB02 — Potsdam Quantitative Eval (all 6 tiles)

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam ISPRS GeoTIFF + labels

**What this does:**
1. Installs env (conda + mmcv + mmseg, ~15 min)
2. Clones `HarishDeepak/rg-segearth-ov3` (our fork, aligned to teammate's protocol)
3. Runs `eval.py configs/cfg_potsdam.py` **separately per tile** (6 runs) to get per-tile metrics
4. Runs `eval.py` once more with **all 6 tiles staged together** to get the pooled/combined
   metric — this matches teammate's protocol exactly (his run shows `Iter(test) [6/6]`,
   i.e. one pooled confusion matrix across all 6 tiles, not an average of 6 separate scores)
5. Reports both: per-tile table + combined pooled table (mIoU/mAcc/mFscore + *_NoClutter variants)

Tiles: `5_14, 5_15, 6_13, 6_14, 6_15, 7_13`


## 1 — Environment setup (~15 min, run once per session)

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
!conda run -n segearth pip install openpyxl -q

In [ ]:
%%bash
# mmseg 1.2.2 declares mmcv<2.2.0 as max; patch so version check passes
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmcv.ops import point_sample; print("MMCV OPS OK")
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Stage helper

`cfg_potsdam.py` expects:
```
/kaggle/working/PotsdamEval/
  img_dir/val/         ← *_RGB.tif
  ann_dir_indexed/val/ ← *_label_noBoundary.tif (converted to 1-indexed)
```
This cell defines a `stage_tiles(tile_list)` helper that wipes and re-stages the eval
directory for a given set of tiles, then converts RGB labels → 1-indexed.

In [ ]:
import shutil
import numpy as np
from PIL import Image
from pathlib import Path

POTSDAM_INPUT = Path("/kaggle/input/datasets/dummyirl/6isprs")
EVAL_ROOT     = Path("/kaggle/working/PotsdamEval")
ALL_TILES     = ["5_14", "5_15", "6_13", "6_14", "6_15", "7_13"]

RGB_TO_IDX = {
    (255, 255, 255): 1,  # impervious surface
    (  0,   0, 255): 2,  # building
    (  0, 255, 255): 3,  # low vegetation
    (  0, 255,   0): 4,  # tree
    (255, 255,   0): 5,  # car
    (255,   0,   0): 6,  # clutter
}

def stage_tiles(tile_list):
    if EVAL_ROOT.exists():
        shutil.rmtree(EVAL_ROOT)
    img_dst   = EVAL_ROOT / "img_dir/val"
    label_dst = EVAL_ROOT / "ann_dir_indexed/val"
    img_dst.mkdir(parents=True, exist_ok=True)
    label_dst.mkdir(parents=True, exist_ok=True)

    for tile in tile_list:
        img_files   = sorted(POTSDAM_INPUT.rglob(f"*{tile}*_RGB.tif"))
        label_files = sorted(POTSDAM_INPUT.rglob(f"*{tile}*_label_noBoundary.tif"))
        for p in img_files:
            dst = img_dst / p.name
            if not dst.exists(): dst.symlink_to(p)
        for p in label_files:
            dst = label_dst / p.name
            if not dst.exists(): dst.symlink_to(p)

    # Convert RGB labels -> 1-indexed (PotsdamDataset reduce_zero_label=True: 1->0 ... 6->5)
    for lbl_path in sorted(label_dst.iterdir()):
        raw = np.array(Image.open(lbl_path))
        if raw.ndim == 2 and raw.max() <= 6 and raw.min() >= 1:
            continue
        rgb = raw[:, :, :3]
        idx = np.zeros(rgb.shape[:2], dtype=np.uint8)
        for color, cls in RGB_TO_IDX.items():
            mask = (rgb[:,:,0]==color[0]) & (rgb[:,:,1]==color[1]) & (rgb[:,:,2]==color[2])
            idx[mask] = cls
        lbl_path.unlink()
        Image.fromarray(idx).save(lbl_path)

    n_img = len(list(img_dst.iterdir()))
    n_lbl = len(list(label_dst.iterdir()))
    print(f"Staged {tile_list}: {n_img} image(s), {n_lbl} label(s)")
    return n_img, n_lbl

## 4 — Run eval PER TILE (6 separate runs)

In [ ]:
import subprocess, re
from pathlib import Path

def run_eval(tag):
    log_path = Path(f"/kaggle/working/potsdam_eval_log_{tag}.txt")
    cmd = (
        "source /tmp/miniconda/bin/activate segearth && "
        "export MPLBACKEND=Agg && "
        "cd /tmp/SegEarth-OV-3 && "
        "python eval.py configs/cfg_potsdam.py "
        "--cfg-options test_dataloader.dataset.data_root=/kaggle/working/PotsdamEval "
        f"> {log_path} 2>&1"
    )
    subprocess.run(["bash", "-c", cmd])
    return log_path.read_text(errors="replace")

def parse_metrics(log_text):
    """Pull the final summary metrics dict mmseg prints at the end of Runner.test()."""
    metrics = {}
    for key in ["aAcc", "mIoU", "mAcc", "mFscore", "mPrecision", "mRecall",
                "mIoU_NoClutter", "mAcc_NoClutter", "mFscore_NoClutter",
                "mPrecision_NoClutter", "mRecall_NoClutter"]:
        m = re.search(rf"{key}:\s*([\d.]+)", log_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics

per_tile_results = {}
TILES = ["5_14", "5_15", "6_13", "6_14", "6_15", "7_13"]

for tile in TILES:
    print(f"=== Evaluating tile {tile} ===")
    stage_tiles([tile])
    log_text = run_eval(tile)
    metrics = parse_metrics(log_text)
    per_tile_results[tile] = metrics
    print(metrics)
    print()

## 5 — Run eval COMBINED (all 6 tiles staged together, matches teammate's pooled '6/6' protocol)

In [ ]:
print("=== Evaluating ALL 6 TILES COMBINED (pooled) ===")
stage_tiles(TILES)
combined_log = run_eval("combined")
combined_metrics = parse_metrics(combined_log)
print(combined_metrics)

# show tail of combined log (per-class table + summary)
print("\n".join(combined_log.strip().splitlines()[-40:]))

## 6 — Summary table: per-tile + combined

In [ ]:
import pandas as pd

rows = []
for tile, m in per_tile_results.items():
    row = {"tile": tile}
    row.update(m)
    rows.append(row)
row = {"tile": "COMBINED (pooled, matches teammate protocol)"}
row.update(combined_metrics)
rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/nb02_all_tiles_summary.csv", index=False)
print("Saved: /kaggle/working/nb02_all_tiles_summary.csv")
df